---
### Digital Egypt Pioneers Initiative (DEPI) - AI Track
**Graduation Project: Detecting Change in Before-and-After Aerial Images (LEVIR-CD+)**

**فريق العمل (Project Team):**
- **Ahmed Refaat Abd-Elmotelb** (Lead ML Engineer & Pipeline Architect)
- **Ahmed Reda Mohamed** (Deep Learning Specialist & Loss Design)
- **Raeed Youssef Mahfouz** (Data Systems & Augmentation Specialist)
- **Ahmed Mohamed Qotb** (Evaluation Metrics & Quality Assurance Lead)
---
# 01 — فحص وتجهيز الداتا (LEVIR-CD+)

الهدف: تنزيل الداتا عبر `kagglehub`، اكتشاف البنية **بدون افتراض أسماء المجلدات**،
مطابقة صور الزمنين والأقنعة بالمعرف المشترك، فحص الأبعاد/القنوات/القيم،
وإحصاءات التقسيم النقي المعتمد الخالي من التسريب (525 train / 112 val / 348 test)، ثم عرض عينات بصرية.

المنطق الفعلي موجود في `src/data_discovery.py` و`src/config.py` — هذا الدفتر
واجهة تفاعلية له. (البديل غير التفاعلي: `python explore_data.py`)

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():   # kernel cwd may be notebooks/
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from src import config
from src.data_discovery import (build_samples, change_ratio_stats, download_dataset,
                                load_mask, print_tree, split_report, verify_samples)

config.ensure_dirs()
print("project root:", PROJECT_ROOT)

In [ ]:
root = download_dataset()
print_tree(root)

## اكتشاف المجلدات ومطابقة العينات

التصنيف يعتمد كلمات مفتاحية (A/B, image1/image2, mask/label ...) ويطابق الملفات
بجذور الأسماء (file stem). إن فشل الاكتشاف توقف الكود برسالة واضحة، ويمكن
تعبئة `OVERRIDES` في `src/data_discovery.py` يدويًا.

In [ ]:
samples, notes = build_samples(root)
print(f"Total matched samples: {len(samples)}")
for s in samples[:5]:
    print("  e.g.", s.stem, "|", s.split, "|", s.a_path.name, s.b_path.name, s.mask_path.name)
print("\nAssumptions / notes:")
for n in notes:
    print("  -", n)

In [ ]:
ver = verify_samples(samples)
print("image sizes :", ver["sizes"])
print("image modes :", ver["image_modes"])
print("mask values :", ver["mask_unique_values"])
print("problems    :", ver["problems"] or "none")

In [ ]:
report = split_report(samples)
print(f"{'split':>6} | {'n':>5} | {'changed%':>9} | {'unchanged%':>10} | empty masks")
for split, st in report.items():
    print(f"{split:>6} | {st['n_samples']:5d} | {st['changed_frac_mean']*100:8.2f}% | "
          f"{st['unchanged_frac_mean']*100:9.2f}% | {st['empty_masks']}/{st['n_masks']}")

## عينات بصرية

In [ ]:
rng = np.random.default_rng(config.SEED)
train_samples = [s for s in samples if s.split == "train"]
picks = rng.choice(len(train_samples), size=3, replace=False)

fig, axes = plt.subplots(3, 3, figsize=(13, 13))
for row, idx in enumerate(picks):
    s = train_samples[int(idx)]
    a = np.array(Image.open(s.a_path).convert("RGB"))
    b = np.array(Image.open(s.b_path).convert("RGB"))
    m = load_mask(s.mask_path)
    axes[row, 0].imshow(a); axes[row, 0].set_title(f"{s.stem} | t1 (A)")
    axes[row, 1].imshow(b); axes[row, 1].set_title("t2 (B)")
    axes[row, 2].imshow(m, cmap="gray", vmin=0, vmax=1)
    axes[row, 2].set_title(f"change mask ({m.mean()*100:.1f}%)")
    for c in range(3):
        axes[row, c].axis("off")
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "data_samples.png", dpi=110)
plt.show()

In [ ]:
# توزيع نسبة التغيّر لكل عينة (على عينة من 200 قناع)
r = [load_mask(s.mask_path).mean() for s in train_samples[:200]]
plt.figure(figsize=(7, 3.5))
plt.hist(np.array(r) * 100, bins=30, color="#3b6ea5")
plt.xlabel("changed pixels per image (%)")
plt.ylabel("images")
plt.title("Distribution of change ratio (train sample)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()